In [6]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import os
import time

In [7]:
# Constants
RANDOMSTATE = 1082
TRAIN_SIZE = 0.8
DATA_PATH = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
MODEL_NAME = "MLP Classifier (SMOTE)"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"

In [8]:
# Load real data
df = pd.read_csv("../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols).values
labels = df['label']

In [9]:
# Data Preprocessing
if labels.dtype == object or labels.dtype.name == 'category':
    le = LabelEncoder()
    y = le.fit_transform(labels)
else:
    y = labels.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RANDOMSTATE, stratify=y
)

print("\nApplying SMOTE to the training data...")
# Get class distribution
unique, counts = np.unique(y_train, return_counts=True)
class_counts = dict(zip(unique, counts))
majority_class = max(class_counts, key=class_counts.get)
count_of_majority_class = class_counts[majority_class]
print(f"Number of samples in the majority class ({majority_class}) in y_train: {count_of_majority_class}")

# Set up SMOTE sampling strategy for all minority classes
desired_strategy = {cls: count_of_majority_class for cls in unique if cls != majority_class}
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=RANDOMSTATE, sampling_strategy=desired_strategy)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"Resampled training features shape: {X_train_resampled.shape}")
print(f"Resampled training target shape: {y_train_resampled.shape}")
print(f"Class distribution in resampled y_train:\n{pd.Series(y_train_resampled).value_counts(normalize=True)}")


Applying SMOTE to the training data...
Number of samples in the majority class (0) in y_train: 624
Resampled training features shape: (1248, 82)
Resampled training target shape: (1248,)
Class distribution in resampled y_train:
0    0.5
1    0.5
Name: proportion, dtype: float64


In [10]:
# Train the model

model = MLPClassifier(
    hidden_layer_sizes=(100, 50, 25),
    activation='tanh',
    solver='adam',
    random_state=RANDOMSTATE,
    learning_rate='adaptive',
    early_stopping=True,
)

train_start_time = time.time()
model.fit(X_train_resampled, y_train_resampled)
train_time = time.time() - train_start_time

In [11]:
# Evaluate the model

print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Evaluating the model on the test set...
Model Accuracy on Test Set: 86.27%
F1 Score: 0.7991

Classification Report:
              precision    recall  f1-score   support

           0       0.86      1.00      0.93      2500
           1       0.00      0.00      0.00       398

    accuracy                           0.86      2898
   macro avg       0.43      0.50      0.46      2898
weighted avg       0.74      0.86      0.80      2898



In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference


_proba = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None
_timing = time_inference(model.predict, X_test_scaled)

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    family="SMOTE",
    labels=[0, 1],
    class_names=["benign", "malicious"],
    y_proba=_proba,
    y_train=y_train_resampled,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=model.get_params(),
    seed=RANDOMSTATE,
    n_features=X_test.shape[1],
    n_train=len(y_train_resampled),
    timing=_timing,
)
append_row(row, RESULTS_CSV)
print(summary(row))
print(f"Results appended to {RESULTS_CSV}")